In [ ]:

# ============================================================
# CELL 1 — WT vs SHANK3B LFP COMPARISON
# Load animal-level results
# ============================================================

%pip -q install scipy statsmodels

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = Path(
    "/content/drive/MyDrive/Reanalysis/"
    "Our_Analysis/lfp/spontaneous"
)

OUT = BASE / "genotype_comparison" / "WT_vs_Shank3B"
OUT.mkdir(parents=True, exist_ok=True)

wt = pd.read_csv(
    BASE / "WT/processing/WT_LFP_all_animal_results.csv"
)

shank = pd.read_csv(
    BASE / "Shank3B/processing/Shank3B_LFP_all_animal_results.csv"
)

wt["genotype"] = "WT"
shank["genotype"] = "Shank3B"

data_all = pd.concat(
    [wt, shank], ignore_index=True
)

BANDS = [
    "Delta", "Theta", "Alpha-like",
    "Beta", "Low gamma", "High gamma"
]

REGIONS = [
    "DG", "CA1", "CA3", "SUB",
    "VISp", "MOs", "CP", "MRN", "APN"
]

MIN_ANIMALS = 5

# One observation per independent animal and anatomical group.
key = [
    "genotype", "subject",
    "region", "subregion"
]

assert not data_all.duplicated(key).any(), (
    "Duplicate animal-region observations!"
)

# Verify all expected spectral columns exist.
POWER_COLUMNS = [
    f"{band}_{metric}"
    for band in BANDS
    for metric in ["absolute_uv2", "relative_pct"]
]

missing = set(POWER_COLUMNS) - set(data_all.columns)

if missing:
    raise RuntimeError(f"Missing columns: {missing}")

# Count animals separately for each anatomical group.
coverage = (
    data_all.groupby(
        ["region", "subregion", "genotype"]
    )["subject"]
    .nunique()
    .unstack(fill_value=0)
)

for genotype in ["WT", "Shank3B"]:
    if genotype not in coverage.columns:
        coverage[genotype] = 0

coverage["eligible"] = (
    (coverage["WT"] >= MIN_ANIMALS)
    & (coverage["Shank3B"] >= MIN_ANIMALS)
)

eligible_groups = coverage.index[coverage["eligible"]]

if len(eligible_groups) == 0:
    raise RuntimeError("No comparable anatomical groups")

data = (
    data_all
    .set_index(["region", "subregion"])
    .loc[eligible_groups]
    .reset_index()
)

# Save combined data.
data_all.to_csv(
    OUT / "all_animal_data.csv",
    index=False
)

data.to_csv(
    OUT / "combined_animal_data.csv",
    index=False
)

coverage.to_csv(
    OUT / "group_coverage.csv"
)

print("=" * 55)
print("WT vs SHANK3B — DATA READY")
print("=" * 55)

print("WT animals:", data.loc[
    data.genotype == "WT", "subject"
].nunique())

print("Shank3B animals:", data.loc[
    data.genotype == "Shank3B", "subject"
].nunique())

print("Comparable region/subregion groups:", len(eligible_groups))

display(coverage)

# Imports required by subsequent cells (loaded once).
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests


In [ ]:

# ============================================================
# CELL 2 — WT vs SHANK3B STATISTICS
# ============================================================


rows = []

for (region, subregion), group in data.groupby(
    ["region", "subregion"]
):
    analysis_level = (
        "region" if subregion == "ALL"
        else "subregion"
    )

    for metric in ["absolute_uv2", "relative_pct"]:
        for band in BANDS:

            column = f"{band}_{metric}"

            x = group.loc[
                group.genotype == "WT",
                column
            ].dropna().to_numpy(dtype=float)

            y = group.loc[
                group.genotype == "Shank3B",
                column
            ].dropna().to_numpy(dtype=float)

            if min(len(x), len(y)) < MIN_ANIMALS:
                continue

            statistic, p = mannwhitneyu(
                x, y,
                alternative="two-sided",
                method="auto"
            )

            # Positive delta = higher Shank3B values.
            cliffs_delta = (
                np.sum(y[:, None] > x[None, :])
                - np.sum(y[:, None] < x[None, :])
            ) / (len(x) * len(y))

            rows.append({
                "analysis_level": analysis_level,
                "region": region,
                "subregion": subregion,
                "band": band,
                "metric": metric,
                "n_WT": len(x),
                "n_Shank3B": len(y),
                "WT_median": np.median(x),
                "Shank3B_median": np.median(y),
                "MW_U": statistic,
                "p_raw": p,
                "cliffs_delta_Shank3B_vs_WT": cliffs_delta
            })

results = pd.DataFrame(rows)

if results.empty:
    raise RuntimeError("No statistical comparisons")

# FDR within the same four families as previous analyses.
results["p_FDR"] = np.nan

for _, idx in results.groupby(
    ["analysis_level", "metric"]
).groups.items():

    results.loc[idx, "p_FDR"] = multipletests(
        results.loc[idx, "p_raw"],
        method="fdr_bh"
    )[1]

results["significant_raw"] = (
    results["p_raw"] < 0.05
)

results["significant_FDR"] = (
    results["p_FDR"] < 0.05
)

results = results.sort_values(
    ["p_FDR", "p_raw"]
).reset_index(drop=True)

results.to_csv(
    OUT / "WT_vs_Shank3B_LFP_statistics.csv",
    index=False
)

print("=" * 60)
print("WT vs SHANK3B — STATISTICAL RESULTS")
print("=" * 60)

print("Total comparisons:", len(results))
print("Raw p < 0.05:", int(results.significant_raw.sum()))
print("FDR p < 0.05:", int(results.significant_FDR.sum()))

print("\nTop 25 comparisons:")
display(
    results.head(25)
)

print("\nFDR-significant findings:")
display(
    results.loc[results.significant_FDR]
)

# Check the particular region implicated in Cntnap2.
print("\nMOs2/3 — HIGH GAMMA:")
display(
    results.loc[
        (results.region == "MOs")
        & (results.subregion == "MOs2/3")
        & (results.band == "High gamma")
    ]
)


In [ ]:

# ============================================================
# CELL 3 — REGIONAL EFFECT-SIZE HEATMAPS
# ============================================================

for metric, label, filename in [
    ("absolute_uv2", "Absolute power", "absolute"),
    ("relative_pct", "Relative power", "relative")
]:

    selected = results.loc[
        (results["metric"] == metric)
        & (results["analysis_level"] == "region")
    ]

    matrix = selected.pivot(
        index="region",
        columns="band",
        values="cliffs_delta_Shank3B_vs_WT"
    ).reindex(
        index=REGIONS,
        columns=BANDS
    )

    fig, ax = plt.subplots(figsize=(11, 6))

    im = ax.imshow(
        matrix.values,
        cmap="RdBu_r",
        vmin=-1,
        vmax=1,
        aspect="auto"
    )

    ax.set_xticks(np.arange(len(BANDS)))
    ax.set_xticklabels(
        BANDS,
        rotation=30,
        ha="right"
    )

    ax.set_yticks(np.arange(len(REGIONS)))
    ax.set_yticklabels(REGIONS)

    for i, region in enumerate(REGIONS):
        for j, band in enumerate(BANDS):

            hit = selected.loc[
                (selected.region == region)
                & (selected.band == band)
            ]

            if hit.empty:
                continue

            p_raw = float(hit.iloc[0]["p_raw"])
            p_fdr = float(hit.iloc[0]["p_FDR"])

            marker = (
                "**" if p_fdr < 0.05
                else "*" if p_raw < 0.05
                else ""
            )

            if marker:
                ax.text(
                    j, i,
                    marker,
                    ha="center",
                    va="center",
                    fontsize=16,
                    fontweight="bold",
                    color="black"
                )

    ax.set_title(
        f"WT vs Shank3B — {label}\n"
        "Cliff's delta (Shank3B vs WT)"
    )

    plt.colorbar(
        im,
        ax=ax,
        label="Cliff's delta"
    )

    fig.text(
        0.5, 0.005,
        "* uncorrected p < 0.05    "
        "** BH-FDR p < 0.05",
        ha="center",
        fontsize=9
    )

    plt.tight_layout(rect=[0, 0.04, 1, 1])

    save_path = OUT / f"heatmap_{filename}.png"

    plt.savefig(
        save_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    print("Saved:", save_path)


In [ ]:

# ============================================================
# CELL 4 — SUBREGION / CORTICAL LAYER HEATMAPS
# ============================================================

for metric, label, filename in [
    ("absolute_uv2", "Absolute power", "absolute"),
    ("relative_pct", "Relative power", "relative")
]:

    selected = results.loc[
        (results["metric"] == metric)
        & (results["analysis_level"] == "subregion")
        & (results["subregion"] != results["region"])
    ].copy()

    selected["label"] = selected["subregion"]

    subregion_order = (
        selected[["region", "subregion"]]
        .drop_duplicates()
        .sort_values(["region", "subregion"])
        ["subregion"]
        .tolist()
    )

    matrix = selected.pivot(
        index="label",
        columns="band",
        values="cliffs_delta_Shank3B_vs_WT"
    ).reindex(
        index=subregion_order,
        columns=BANDS
    )

    fig, ax = plt.subplots(
        figsize=(11, max(6, len(matrix) * 0.42 + 1.5))
    )

    im = ax.imshow(
        matrix.values,
        cmap="RdBu_r",
        vmin=-1,
        vmax=1,
        aspect="auto"
    )

    ax.set_xticks(range(len(BANDS)))
    ax.set_xticklabels(
        BANDS, rotation=30, ha="right"
    )

    ax.set_yticks(range(len(matrix)))
    ax.set_yticklabels(matrix.index)

    for i, subregion in enumerate(matrix.index):
        for j, band in enumerate(BANDS):

            hit = selected.loc[
                (selected["subregion"] == subregion)
                & (selected["band"] == band)
            ]

            if hit.empty:
                continue

            p_raw = float(hit.iloc[0]["p_raw"])
            p_fdr = float(hit.iloc[0]["p_FDR"])

            marker = (
                "**" if p_fdr < 0.05
                else "*" if p_raw < 0.05
                else ""
            )

            if marker:
                ax.text(
                    j, i,
                    marker,
                    ha="center",
                    va="center",
                    fontsize=15,
                    fontweight="bold",
                    color="black"
                )

    ax.set_title(
        f"WT vs Shank3B — Subregions\n{label}"
    )

    plt.colorbar(
        im,
        ax=ax,
        label="Cliff's delta (Shank3B vs WT)"
    )

    fig.text(
        0.5, 0.005,
        "* uncorrected p < 0.05    "
        "** BH-FDR p < 0.05",
        ha="center",
        fontsize=9
    )

    plt.tight_layout(rect=[0, 0.04, 1, 1])

    save_path = OUT / f"subregions_{filename}.png"

    plt.savefig(
        save_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
